# Day 1 — Ingestion

Goal: download a GitHub docs repo, parse each markdown file's frontmatter, and
get everything into a list of dicts I can work with. My corpus is
`dbt-labs/docs.getdbt.com` — dbt's own documentation.

**Repo gotcha:** the default branch is `current`, not `main`. The course code
fetches `.../heads/main`, which on this repo is an older snapshot. I use
`current` to get what's actually live on docs.getdbt.com.

In [1]:
import io
import zipfile

import requests
import frontmatter

## 1. First pass — download and parse inline

Getting the mechanics working before wrapping it in a function.
`codeload.github.com/.../zip/refs/heads/<branch>` returns the repo as a zip.

In [2]:
url = "https://codeload.github.com/dbt-labs/docs.getdbt.com/zip/refs/heads/current"
resp = requests.get(url)
print("status:", resp.status_code, "| size:", f"{len(resp.content)/1_000_000:.0f} MB")

status: 200 | size: 549 MB


In [3]:
repository_data = []

# Create a ZipFile object from the downloaded content
zf = zipfile.ZipFile(io.BytesIO(resp.content))

for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files (.md and .mdx)
    if not (filename.endswith(".md") or filename.endswith(".mdx")):
        continue

    # Read and parse each file: frontmatter.loads splits the YAML header
    # (--- title: ... ---) from the body
    with zf.open(file_info) as f_in:
        content = f_in.read().decode("utf-8", errors="ignore")
        post = frontmatter.loads(content)
        data = post.to_dict()          # frontmatter fields become dict keys
        data["filename"] = file_info.filename
        repository_data.append(data)

zf.close()

## 2. Explore what I got

In [4]:
len(repository_data)

1547

Each record is a dict: the frontmatter fields (`title`, `description`, etc.)
plus a `content` key holding the markdown body, plus the `filename` I added.

In [5]:
repository_data[100]

{'title': 'To defer or to clone, that is the question',
 'description': 'In dbt v1.6, we introduce support for zero-copy cloning via the new dbt clone command. In this blog post, Kshitij will cover what clone is, how it is different from deferral, and when to use each.',
 'slug': 'to-defer-or-to-clone',
 'image': '/img/blog/2023-10-31-to-defer-or-to-clone/preview.png',
 'authors': ['kshitij_aranke', 'doug_beatty'],
 'tags': ['analytics craft'],
 'hide_table_of_contents': False,
 'date': datetime.date(2023, 10, 31),
 'is_featured': True,
 'content': "Hi all, I’m Kshitij, a senior software engineer on the Core team at dbt Labs.\nOne of the coolest moments of my career here thus far has been shipping the new `dbt clone` command as part of the dbt-core v1.6 release.\n\nHowever, one of the questions I’ve received most frequently is guidance around “when” to clone that goes beyond [the documentation on “how” to clone](https://docs.getdbt.com/reference/commands/clone).\nIn this blog post, I’l

In [6]:
# peek at the first couple of records
for i in range(2):
    print(repository_data[i].get("title"), "|", repository_data[i]["filename"])

None | docs.getdbt.com-current/.agents/skills/cavecrew/README.md
None | docs.getdbt.com-current/.agents/skills/cavecrew/SKILL.md


## 3. Homework check — find files with "wizard" in the name

dbt shipped an AI product called Wizard, so it shows up a lot. Lowercasing both
sides matters — a plain `in` check is case-sensitive and would miss `Wizard`.

In [7]:
[d["filename"] for d in repository_data if "wizard" in d["filename"].lower()]

['docs.getdbt.com-current/website/blog/2026-06-25-wizard-use-cases.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-1-intro.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-2-understand-project.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-3-validate-changes.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-4-data-informed-tests.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-5-debug-failed-job.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-6-production-deferral.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-7-semantic-layer.md',
 'docs.getdbt.com-current/website/docs/best-practices/how-to-use-wizard/wizard-8-plugins-hooks.md',
 'docs.getdbt.com-current/website/docs/docs/dbt-ai/_wizard-cli-full-generated.md',
 'docs.getdbt.com-current/web

## 4. Wrap it in a reusable function

The course parameterises owner/repo/branch so the same code works on any repo.
`branch='main'` stays the default (works for most repos); I pass `'current'`
for this one. I also decode as UTF-8 and wrap each file in try/except so one
bad file doesn't kill the whole ingest.

In [8]:
def read_repo_data(repo_owner, repo_name, branch="main"):
    """
    Download and parse all markdown files from a GitHub repository.

    Args:
        repo_owner: GitHub username or organization
        repo_name:  Repository name
        branch:     Branch to fetch (default 'main'; this repo needs 'current')

    Returns:
        List of dictionaries containing file content and metadata
    """
    url = f"https://codeload.github.com/{repo_owner}/{repo_name}/zip/refs/heads/{branch}"
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith(".md") or filename_lower.endswith(".mdx")):
            continue

        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode("utf-8", errors="ignore")
                post = frontmatter.loads(content)
                data = post.to_dict()
                data["filename"] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue

    zf.close()
    return repository_data

In [9]:
dbt_labs = read_repo_data("dbt-labs", "docs.getdbt.com", branch="current")
print(len(dbt_labs))

1547


## Day 1 done

`dbt_labs` is my parsed corpus — ~1,547 documents, each with frontmatter and a
`content` body. Day 2 picks up from here: `content` is what gets chunked.

**Note on the count:** `current` gives ~1,547; `main` would give fewer (it lags).
dbt commits daily, so the exact number drifts — anything close is correct as long
as I know which branch produced it.